This predicts target speed

In [ ]:
import numpy as np
from numpy import genfromtxt
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import callbacks, optimizers

In [ ]:
os.chdir('#redacted')

raw_data = genfromtxt('model_features6.csv', delimiter=',')

raw_data = np.delete(raw_data,[0],axis = 0)

target = raw_data[:,1]

raw_data = np.delete(raw_data,[1],axis = 1)

raw_data[0:4,0:4]

In [ ]:
target[0]

In [ ]:
num_train_samples = int(0.5 * len(raw_data))
num_val_samples = int(0.25 * len(raw_data))
num_test_samples = len(raw_data) - num_train_samples - num_val_samples
print("num_train_samples:", num_train_samples)
print("num_val_samples:", num_val_samples)
print("num_test_samples:", num_test_samples)

In [ ]:
print(raw_data[0:4,0:4])
print(raw_data[0:4,0:4].mean(axis=0))
print(raw_data.mean(axis=0))

In [ ]:
#mean = raw_data[:num_train_samples].mean(axis=0)
#raw_data -= mean
#std = raw_data[:num_train_samples].std(axis=0)
#raw_data /= std

sampling_rate = 6
sequence_length = 120
delay = sampling_rate * (sequence_length + 24 - 1)
batch_size = 256

train_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=target[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=False,
    batch_size=batch_size,
    start_index=0,
    end_index=num_train_samples)

val_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=target[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=False,
    batch_size=batch_size,
    start_index=num_train_samples,
    end_index=num_train_samples + num_val_samples)

test_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=target[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=False,
    batch_size=batch_size,
    start_index=num_train_samples + num_val_samples)

for samples, targets in train_dataset:
 print("samples shape:", samples.shape)
print("targets shape:", targets.shape)

In [ ]:
input_shape_ = (120, 41) #this is the size of the input matrix.  It doesn't include the batch size.
lag_length = 25
model = tf.keras.Sequential([
    # A dropout removed the specified fraction of the training data randomly to prevent over-fitting

    # keras_init = initializers.RandomNormal(mean=0.,stddev=1.)

    # This is an RNN layer with 128 GRU units representing the dimensionality of the output space
tf.keras.layers.GRU(128, return_sequences=True, input_shape=input_shape_),
tf.keras.layers.Dense(lag_length, activation="relu"),
tf.keras.layers.GRU(128, return_sequences=True, dropout=0.1),
tf.keras.layers.Dense(lag_length, activation="relu"),
tf.keras.layers.GRU(128, return_sequences=False),

    # This is a densely-connected neural network layer with a sequence length of the original dataset
tf.keras.layers.Dense(1, activation="relu")])

adam_ = optimizers.Adam(learning_rate=0.001)

    # This is the loss-function metric used against the validation set (mean absolute error)
model.compile(loss="mae", optimizer=adam_)

model.summary()

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("jena_lstm.keras",
                                    save_best_only=True)
]
history = model.fit(train_dataset,
                    epochs=10,
                    validation_data=val_dataset,
                    callbacks=callbacks)

In [ ]:
raw_data[0:120,:]